# Oceňování opcí

*Nad rámec skripta — aplikace.*

---

**Opce** je právo (ne povinnost) koupit akcii za předem danou cenu $K$ v daném čase $T$. Jestli ho využijete, poznáte až na konci: když akcie stojí víc než $K$, koupíte levně a hned prodáte draho; když míň, právo propadne. Výplata je tedy

$$\max(S_T - K,\ 0),$$

a cena opce je střední hodnota téhle výplaty, odúročená na dnešek. Což je **přesně úloha pro Monte Carlo** — střední hodnota funkce náhodné veličiny.

Tenhle sešit je v repozitáři zvláštní dvěma věcmi:

1. **Má přesnou kontrolu.** Pro evropskou opci existuje **Blackův–Scholesův vzorec**, takže si můžeme ověřit odhad na šest platných cifer.
2. **Je to úloha, kde se snižování rozptylu ze cvičení 7 konečně opravdu vyplatí.** U asijské opce dostaneme pomocí kontrolní proměnné zlepšení v řádu **stovek**, ne procent.

Uvidíte:

- model ceny akcie a Blackův–Scholesův vzorec jako kontrola
- **antitetické proměnné** na evropské opci
- **asijská opce**, kde vzorec neexistuje — a kontrolní proměnná, která má vzorec
- **bariérová opce**, kde záleží na celé cestě

In [ ]:
import math
import random

import matplotlib.pyplot as plt

random.seed(1)  # opakovatelné výsledky


def Phi(x):
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))


def estimate(xs):
    n = len(xs)
    m = sum(xs) / n
    var = sum((x - m) ** 2 for x in xs) / (n - 1)
    return m, math.sqrt(var / n)


def check(name, mean, se, exact, k=5):
    print(f"{name}: {mean:.6f} +- {se:.6f}  (přesně {exact:.6f})")
    assert abs(mean - exact) <= k * se, name


# --- parametry trhu ---
S0 = 100.0     # dnešní cena akcie
K_ = 100.0     # realizační cena opce
R = 0.03       # bezriziková úroková míra
SIGMA = 0.25   # volatilita
T = 1.0        # doba do splatnosti v letech
KROKU = 50     # kolikrát cestu sledujeme (pro asijskou a bariérovou)

## 1. Model ceny akcie

Standardní model (geometrický Brownův pohyb) říká, že **logaritmus** ceny se vyvíjí jako náhodná procházka s driftem:

$$S_T = S_0 \exp\!\Big( \big(r - \tfrac{\sigma^2}{2}\big)T + \sigma\sqrt{T}\,Z \Big), \qquad Z \sim N(0,1).$$

Proč logaritmus: cena nesmí být záporná a změny jsou **násobné** (akcie spadne o 10 %, ne o 10 korun). Je to první **multiplikativní** proces v repozitáři — všechno ostatní bylo sčítání.

Cena opce je odúročená střední hodnota výplaty:

$$C = e^{-rT}\,E\big[\max(S_T - K, 0)\big].$$

Pro tenhle případ se integrál dá spočítat ručně a vyjde **Blackův–Scholesův vzorec**

$$C = S_0\,\Phi(d_1) - K e^{-rT}\Phi(d_2), \qquad
d_1 = \frac{\ln(S_0/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}, \quad d_2 = d_1 - \sigma\sqrt{T}.$$

Máme tedy vzácnou věc: úlohu pro Monte Carlo, kde **známe přesnou odpověď**.

In [ ]:
def black_scholes(S0, K, r, sig, T):
    """Přesná cena evropské kupní opce."""
    d1 = (math.log(S0 / K) + (r + sig * sig / 2) * T) / (sig * math.sqrt(T))
    d2 = d1 - sig * math.sqrt(T)
    return S0 * Phi(d1) - K * math.exp(-r * T) * Phi(d2)


def cena_na_konci(z=None):
    """S_T z jednoho normálního čísla."""
    z = random.gauss(0, 1) if z is None else z
    return S0 * math.exp((R - SIGMA**2 / 2) * T + SIGMA * math.sqrt(T) * z)


PRESNE = black_scholes(S0, K_, R, SIGMA, T)
print(f"Black–Scholes: {PRESNE:.6f}")

n = 200_000
vyplaty = [math.exp(-R * T) * max(cena_na_konci() - K_, 0.0) for _ in range(n)]
check("Monte Carlo", *estimate(vyplaty), PRESNE)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
ax = axes[0]
konce = [cena_na_konci() for _ in range(50_000)]
w = 250 / 100
ax.hist(konce, bins=100, range=(0, 250), weights=[1 / (len(konce) * w)] * len(konce), alpha=0.6)
ax.axvline(K_, color="tab:red", ls="--", label=f"realizační cena $K$ = {K_:.0f}")
ax.set_xlabel("$S_T$")
ax.set_ylabel("hustota")
ax.set_title("Cena na konci je lognormální", fontsize=10)
ax.legend(fontsize=8)

ax = axes[1]
for _ in range(40):
    dt = T / KROKU
    S, cesta = S0, [S0]
    for _ in range(KROKU):
        S *= math.exp((R - SIGMA**2 / 2) * dt + SIGMA * math.sqrt(dt) * random.gauss(0, 1))
        cesta.append(S)
    ax.plot(cesta, lw=0.7, alpha=0.7, color="tab:green" if cesta[-1] > K_ else "tab:red")
ax.axhline(K_, color="k", ls="--", lw=1)
ax.set_xlabel("krok")
ax.set_ylabel("cena akcie")
ax.set_title("40 cest (zelená = opce se vyplatí)", fontsize=10)
plt.tight_layout()
plt.show()

## 2. Antitetické proměnné

Výplata $\max(S_T - K, 0)$ je **rostoucí** funkcí normálního čísla $Z$. To je přesně situace, kdy podle cvičení 7 pomáhají **antitetické proměnné**: místo dvou nezávislých $Z$ použijeme dvojici $(Z, -Z)$, která je záporně korelovaná.

In [ ]:
def evropska_antiteticky(n_dvojic):
    hodnoty = []
    for _ in range(n_dvojic):
        z = random.gauss(0, 1)
        a = math.exp(-R * T) * max(cena_na_konci(z) - K_, 0.0)
        b = math.exp(-R * T) * max(cena_na_konci(-z) - K_, 0.0)
        hodnoty.append((a + b) / 2)
    return hodnoty


N = 100_000
obycejne = [math.exp(-R * T) * max(cena_na_konci() - K_, 0.0) for _ in range(2 * N)]
anti = evropska_antiteticky(N)          # stejný počet vyhodnocení!

m1, se1 = estimate(obycejne)
m2, se2 = estimate(anti)
se2 = se2 / math.sqrt(1)                # anti má N dvojic = 2N vyhodnocení
print(f"{'metoda':>24s} {'odhad':>11s} {'sm. chyba':>11s}")
print(f"{'obyčejné MC':>24s} {m1:>11.6f} {se1:>11.6f}")
print(f"{'antitetické':>24s} {m2:>11.6f} {se2:>11.6f}")
print(f"{'přesně':>24s} {PRESNE:>11.6f}")
print(f"\nzisk: {(se1 / se2) ** 2:.1f}x menší rozptyl při stejném počtu vyhodnocení")
assert abs(m2 - PRESNE) < 5 * se2

## 3. Asijská opce: když vzorec neexistuje

**Asijská opce** se nevyplácí podle konečné ceny, ale podle **průměru** ceny za celé období:

$$\max\!\left(\frac{1}{n}\sum_{i=1}^{n} S_{t_i} - K,\ 0\right).$$

Používá se proto, že se s ní hůř manipuluje — nestačí pohnout cenou v jediný den.

Pro aritmetický průměr lognormálních veličin **žádný uzavřený vzorec neexistuje** (součet lognormálních není lognormální). Monte Carlo ale funguje úplně stejně.

### Kontrolní proměnná, která vzorec má

A teď ten trik. Nahradíme-li aritmetický průměr **geometrickým**,

$$G = \Big(\prod_i S_{t_i}\Big)^{1/n} = \exp\!\Big(\tfrac1n \textstyle\sum_i \ln S_{t_i}\Big),$$

je $\ln G$ součtem normálních veličin, tedy **normální** — a cena takové opce se spočítat dá. Přitom je geometrická asijská opce s tou aritmetickou skoro dokonale korelovaná.

To je přesně kontrolní proměnná ze cvičení 7: veličina blízká té naší, jejíž střední hodnotu známe přesně.

Konkrétně $\ln G \sim N(\mu_G, \sigma_G^2)$ s

$$\mu_G = \ln S_0 + \Big(r - \tfrac{\sigma^2}{2}\Big)\frac{T(n+1)}{2n}, \qquad
\sigma_G^2 = \sigma^2 T\,\frac{(n+1)(2n+1)}{6n^2},$$

a cena vyjde ze vzorce pro lognormální veličinu.

In [ ]:
def geometricka_asijska_presne(S0, K, r, sig, T, n):
    """Přesná cena geometrické asijské opce."""
    mu_G = math.log(S0) + (r - sig * sig / 2) * T * (n + 1) / (2 * n)
    var_G = sig * sig * T * (n + 1) * (2 * n + 1) / (6 * n * n)
    sd_G = math.sqrt(var_G)
    d1 = (mu_G - math.log(K) + var_G) / sd_G
    d2 = d1 - sd_G
    return math.exp(-r * T) * (math.exp(mu_G + var_G / 2) * Phi(d1) - K * Phi(d2))


def cesta():
    """Jedna cesta; vrací (aritmetický průměr, geometrický průměr, minimum)."""
    dt = T / KROKU
    S, soucet, soucet_log, minimum = S0, 0.0, 0.0, S0
    for _ in range(KROKU):
        S *= math.exp((R - SIGMA**2 / 2) * dt + SIGMA * math.sqrt(dt) * random.gauss(0, 1))
        soucet += S
        soucet_log += math.log(S)
        minimum = min(minimum, S)
    return soucet / KROKU, math.exp(soucet_log / KROKU), minimum


G_PRESNE = geometricka_asijska_presne(S0, K_, R, SIGMA, T, KROKU)
print(f"geometrická asijská, přesný vzorec: {G_PRESNE:.6f}")

n = 100_000
ar, ge = [], []
for _ in range(n):
    a, g, _ = cesta()
    ar.append(math.exp(-R * T) * max(a - K_, 0.0))
    ge.append(math.exp(-R * T) * max(g - K_, 0.0))

check("geometrická asijská, MC", *estimate(ge), G_PRESNE)   # kontrola vzorce simulací

m_ar, se_ar = estimate(ar)
print(f"\naritmetická asijská, obyčejné MC: {m_ar:.6f} +- {se_ar:.6f}")

# kontrolní proměnná
m_g = sum(ge) / n
cov = sum((x - m_ar) * (y - m_g) for x, y in zip(ar, ge)) / (n - 1)
var_g = sum((y - m_g) ** 2 for y in ge) / (n - 1)
c_opt = cov / var_g
korelace = cov / math.sqrt(var_g * sum((x - m_ar) ** 2 for x in ar) / (n - 1))
upraveno = [x - c_opt * (y - G_PRESNE) for x, y in zip(ar, ge)]
m_cv, se_cv = estimate(upraveno)
print(f"aritmetická asijská, kontrolní:   {m_cv:.6f} +- {se_cv:.6f}")
print(f"\nkorelace aritmetické a geometrické: {korelace:.5f}   (optimální c = {c_opt:.3f})")
print(f"zisk: {(se_ar / se_cv) ** 2:.0f}x menší rozptyl")
print(f"tedy obyčejné MC by na stejnou přesnost potřebovalo {(se_ar / se_cv) ** 2:.0f}x víc cest")

Korelace mezi aritmetickou a geometrickou verzí je přes 0.999, a zisk je proto v řádu stovek. Tohle je nejsilnější ukázka kontrolní proměnné v celém repozitáři — ve cvičení 7 jsme na umělém příkladu dostali zhruba 66×, tady vychází **několik set**.

Důvod je obecný a stojí za zapamatování: **dobrá kontrolní proměnná je tatáž úloha, jen o kousek jednodušší.** Nehledá se podobná funkce, hledá se zjednodušení, které ještě má vzorec.

## 4. Bariérová opce: záleží na celé cestě

Poslední varianta: opce, která **propadne**, pokud cena akcie někdy během života klesne pod bariéru $B$ (down-and-out). Výplata je

$$\max(S_T - K, 0)\cdot \mathbf 1\big[\min_t S_t > B\big].$$

Tady je na simulaci cest vidět to hlavní: nejde o rozdělení konečné ceny, ale o **celou trajektorii**. Analytický vzorec existuje, ale jen pro spojité sledování — my sledujeme v $n$ okamžicích, což dává jinou (a pro skutečné kontrakty realističtější) cenu.

In [ ]:
def cesta_s_koncem():
    dt = T / KROKU
    S, minimum = S0, S0
    for _ in range(KROKU):
        S *= math.exp((R - SIGMA**2 / 2) * dt + SIGMA * math.sqrt(dt) * random.gauss(0, 1))
        minimum = min(minimum, S)
    return S, minimum


print(f"{'bariéra B':>11s} {'cena opce':>22s} {'podíl ceny bez bariéry':>24s}")
bariery = (0, 60, 70, 80, 90, 95)
ceny = []
for B in bariery:
    hod = []
    for _ in range(80_000):
        ST, mn = cesta_s_koncem()
        hod.append(math.exp(-R * T) * max(ST - K_, 0.0) if mn > B else 0.0)
    m, se = estimate(hod)
    ceny.append(m)
    print(f"{B:>11d} {m:>14.5f} +- {se:.5f} {m / PRESNE:>23.1%}")

print(f"\nB = 0 znamená žádnou bariéru, takže musí vyjít Black–Scholes ({PRESNE:.4f}) — a vyjde.")

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot(bariery, ceny, "o-")
ax.axhline(PRESNE, color="tab:red", ls="--", label=f"bez bariéry = Black–Scholes ({PRESNE:.3f})")
ax.set_xlabel("bariéra $B$")
ax.set_ylabel("cena opce")
ax.set_title("Čím blíž je bariéra, tím levnější (a riskantnější) opce", fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Shrnutí

| Pojem | Co si pamatovat |
|-------|-----------------|
| Cena opce | $e^{-rT}E[\text{výplata}]$ — střední hodnota, tedy úloha pro MC |
| Model ceny | multiplikativní: $S_T = S_0 e^{(r-\sigma^2/2)T + \sigma\sqrt T Z}$ |
| Black–Scholes | přesná cena evropské opce — vzácná možnost ověřit MC na 6 cifer |
| Antitetické | výplata je monotónní v $Z$, takže dvojice $(Z, -Z)$ pomáhá |
| Asijská opce | průměr místo konce; **aritmetická nemá vzorec**, geometrická ano |
| Kontrolní proměnná | geometrická asijská: korelace > 0.999 ⟹ zisk ve stovkách |
| Bariérová opce | rozhoduje celá cesta, ne jen konec; $B = 0$ vrací Black–Scholes |

Hlavní myšlenka: **nejlepší kontrolní proměnná je zjednodušená verze téže úlohy.** Nehledejte podobnou funkci — hledejte, co se v úloze dá zjednodušit tak, aby ještě šla spočítat přesně.

### Zkuste si

1. Spočítejte **prodejní** opci (put, výplata $\max(K - S_T, 0)$) a ověřte paritu $C - P = S_0 - Ke^{-rT}$.
2. Přidejte ke kontrolní proměnné ještě antitetické vzorkování. Násobí se zisky, nebo se překrývají?
3. Jak závisí zisk z kontrolní proměnné na počtu kroků $n$? Zkuste $n = 5$ a $n = 500$.
4. Spočítejte **deltu** opce (derivaci ceny podle $S_0$) konečnou diferencí. Proč je potřeba použít **tatáž** náhodná čísla pro obě ceny? (Nápověda: společné náhodné veličiny jsou další metoda snižování rozptylu.)
5. Bariérová opce se sleduje jen v $n$ okamžicích. Jak se cena mění s rostoucím $n$? K čemu konverguje?
6. Nahraďte normální rozdělení výnosů těžším chvostem (třeba Studentovým). O kolik se změní cena opce hluboko mimo peníze?